In [1]:
pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 35.2 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
from transformers import pipeline

pipe = pipeline("text-generation", model="ibm-granite/granite-3.3-2b-instruct")
messages = [
    {"role": "user", "content": "who are you?"},
]

pipe(messages)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

...

added_tokens.json:   0%|          | 0.00/207 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': [{'role': 'user', 'content': 'who are you?'},
   {'role': 'assistant',
    'content': 'I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.'}]}]

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-3.3-2b-instruct")
messages = [
{"role": "user", "content": "who are you?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

I am an assistant designed to help answer your questions and provide information. I strive to provide concise and accurate responses in English.


In [4]:
pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 19.5 MB/s eta 0:00:00


In [5]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

import gradio as gr
from datetime import datetime
from pathlib import Path
import json
import sqlite3
import re
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import requests
import urllib.parse

In [6]:
# ============================================================================
# DATABASE SCHEMA
# ============================================================================
SCHEMA = {
'users': {
'id': 'INTEGER PRIMARY KEY',
'name': 'TEXT',
'email': 'TEXT',
'signup_date': 'DATE',
'age': 'INTEGER',
'country': 'TEXT',
'status': 'TEXT'
},
'orders': {
'id': 'INTEGER PRIMARY KEY',
'user_id': 'INTEGER',
'product_name': 'TEXT',
'amount': 'REAL',
'order_date': 'DATE',
'status': 'TEXT'
},
'products': {
'id': 'INTEGER PRIMARY KEY',
'name': 'TEXT',
'price': 'REAL',
'category': 'TEXT',
'stock': 'INTEGER'
},
'transactions': {
'id': 'INTEGER PRIMARY KEY',
'user_id': 'INTEGER',
'amount': 'REAL',
'date': 'DATE',
'type': 'TEXT'
}
}

In [7]:
# ============================================================================
# SECURITY & SQL GENERATION
# ============================================================================
DANGEROUS_KEYWORDS = ['DROP', 'DELETE', 'ALTER', 'TRUNCATE', 'INSERT', 'UPDATE', 'EXEC', 'EXECUTE']
DANGEROUS_CHARS = [';', '--', '/', '/', 'xp_', 'sp_']

def is_safe(text):
    """Check if text contains dangerous SQL keywords"""
    text_upper = text.upper()
    for keyword in DANGEROUS_KEYWORDS:
        if keyword in text_upper:
            return False
    for char in DANGEROUS_CHARS:
        if char in text:
            return False
    return True

def get_table_name(query_text):
    """Intelligently infer table name from natural language"""
    query_lower = query_text.lower()

    keyword_map = {
        'users': ['user', 'users', 'customer', 'customers', 'account', 'accounts', 'profile'],
        'orders': ['order', 'orders', 'purchase', 'purchases', 'buy', 'bought'],
        'products': ['product', 'products', 'item', 'items', 'goods'],
        'transactions': ['transaction', 'transactions', 'payment', 'payments', 'transfer']
    }

    for table, keywords in keyword_map.items():
        for keyword in keywords:
            if keyword in query_lower:
                return table

    return 'users' # Default table

In [8]:
def get_column_names(table_name, query_text):
    """Map natural language to actual column names"""
    query_lower = query_text.lower()
    columns = list(SCHEMA[table_name].keys())
    selected_cols = []

    column_map = {
        'name': ['name', 'username', 'full name'],
        'email': ['email', 'mail', 'address'],
        'signup_date': ['signup', 'created', 'joined', 'registered', 'date', 'month'],
        'age': ['age', 'years'],
        'amount': ['amount', 'price', 'cost', 'total'],
        'date': ['date', 'when'],
        'status': ['status', 'state'],
        'country': ['country', 'location', 'place']
    }

    for col, keywords in column_map.items():
        if col in columns:
            for keyword in keywords:
                if keyword in query_lower:
                    selected_cols.append(col)
                    break

    if not selected_cols:
        selected_cols = ['*']
    else:
        selected_cols = list(dict.fromkeys(selected_cols)) # Remove duplicates

    return selected_cols

In [9]:
def extract_conditions(query_text, table_name):
    """Extract WHERE conditions from natural language"""
    query_lower = query_text.lower()
    conditions = []

    # Determine the appropriate date column for the given table
    date_column = None
    if 'signup_date' in SCHEMA.get(table_name, {}):
        date_column = 'signup_date'
    elif 'order_date' in SCHEMA.get(table_name, {}):
        date_column = 'order_date'
    elif 'date' in SCHEMA.get(table_name, {}):
        date_column = 'date'

    # Specific date patterns (order matters here, more specific first)
    if date_column:
        # This month
        if 'this month' in query_lower:
            conditions.append(f"strftime('%Y-%m', {date_column}) = strftime('%Y-%m', 'now')")

        # Last month (previous calendar month)
        elif 'last month' in query_lower:
            conditions.append(f"strftime('%Y-%m', {date_column}) = strftime('%Y-%m', 'now', '-1 month')")

        # This year
        elif 'this year' in query_lower:
            conditions.append(f"strftime('%Y', {date_column}) = strftime('%Y', 'now')")

        # Last year (previous calendar year)
        elif 'last year' in query_lower:
            conditions.append(f"strftime('%Y', {date_column}) = strftime('%Y', 'now', '-1 year')")

        # Last week
        if 'last week' in query_lower:
            conditions.append(f"DATE({date_column}) >= DATE('now', '-7 days')")

        # Today
        if 'today' in query_lower:
            conditions.append(f"DATE({date_column}) = DATE('now')")

        # Specific year (e.g., 'in 2023')
        year_match = re.search(r'in\s+([0-9]{4})', query_lower)
        if year_match:
            year = year_match.group(1)
            conditions.append(f"strftime('%Y', {date_column}) = '{year}'")


    # General conditions (status, amount, country, name)

    # Active/inactive
    if 'status' in SCHEMA.get(table_name, {}):
        if 'active' in query_lower:
            conditions.append("status = 'active'")
        elif 'inactive' in query_lower:
            conditions.append("status = 'inactive'")

    # Amount filtering
    if 'amount' in SCHEMA.get(table_name, {}):
        amount_match_gt = re.search(r'(?:more than|greater than|above)\s+([0-9]+)', query_lower)
        if amount_match_gt:
            conditions.append(f"amount > {amount_match_gt.group(1)}")

        amount_match_lt = re.search(r'(?:less than|below)\s+([0-9]+)', query_lower)
        if amount_match_lt:
            conditions.append(f"amount < {amount_match_lt.group(1)}")

    # Country filtering (only if the table has a 'country' column)
    if 'country' in SCHEMA.get(table_name, {}):
        country_match = re.search(r'(?:from|in)\s+([a-zA-Z\s]+)', query_lower)
        if country_match:
            potential_country = country_match.group(1).strip().lower()
            # Avoid matching if it looks like a date phrase that wasn't caught by earlier date logic
            if not any(phrase in potential_country for phrase in ['month', 'year', 'week', 'today']):
                country_name = potential_country.title()
                conditions.append(f"country = '{country_name}'")

    # Name filtering (only if the table has a 'name' column)
    if 'name' in SCHEMA.get(table_name, {}):
        found_name_condition = False

        # Try 'LIKE' name match first
        name_like_match = re.search(r'name\s+like\s+([a-zA-Z%]+)', query_lower)
        if name_like_match:
            like_pattern = name_like_match.group(1).strip()
            if not like_pattern.startswith('%'):
                like_pattern = '%' + like_pattern
            if not like_pattern.endswith('%'):
                like_pattern = like_pattern + '%'
            conditions.append(f"name LIKE '{like_pattern}'")
            found_name_condition = True

        # If no 'LIKE' match, try exact name match
        if not found_name_condition:
            name_match = re.search(r'(?:with name|named|name is)\s+([a-zA-Z]+)', query_lower)
            if name_match:
                person_name = name_match.group(1).strip().title()
                conditions.append(f"name = '{person_name}'")

    return conditions

In [10]:
def generate_sql(query_text):
    """Generate SQLite SQL from natural language"""

    if not is_safe(query_text):
        return None, "❌ UNSAFE: Query contains dangerous SQL keywords!"

    table = get_table_name(query_text)
    columns = get_column_names(table, query_text)
    columns_str = ', '.join(columns) if columns != ['*'] else '*'

    # Build base query
    sql = f"SELECT {columns_str} FROM {table}"

    # Add conditions
    conditions = extract_conditions(query_text, table)
    if conditions:
        sql += " WHERE " + " AND ".join(conditions)

    # Add limit
    if 'all' not in query_text.lower():
        sql += " LIMIT 10"

    return sql, f"✅ Generated from table '{table}' with columns: {columns_str}"

In [11]:
query_history = []
def process_query(user_query, chat_history):
    """Process database query request"""

    if not user_query.strip():
        return chat_history

    chat_history.append((user_query, "Processing..."))

    sql_query, explanation = generate_sql(user_query)

    if sql_query is None:
        response = f"❌ ERROR\n\n{explanation}"
    else:
        response = f"✅ SQL GENERATED\n\n```sql\n{sql_query}\n```\n\n📝 {explanation}"
        query_history.append({
            'user': user_query,
            'sql': sql_query,
            'timestamp': datetime.now().isoformat()
        })

    chat_history[-1] = (user_query, response)
    return chat_history

In [12]:
def download_pdf():
    if not query_history:
        return "❌ No queries to export!", None

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"sql_queries_{timestamp}.pdf"

    try:
        doc = SimpleDocTemplate(filename, pagesize=letter)
        styles = getSampleStyleSheet()
        story = []

        title_style = ParagraphStyle(
            'CustomTitle',
            parent=styles['Heading1'],
            fontSize=26,
            textColor='#FF6B6B',
            spaceAfter=30,
            alignment=1
        )

        story.append(Paragraph("🔍 SQL Query Report", title_style))
        story.append(Paragraph(f"Generated: {datetime.now()}", styles['Normal']))
        story.append(Spacer(1, 0.3*inch))

        for i, query in enumerate(query_history, 1):
            story.append(Paragraph(f"<b>Query #{i}</b>", styles['Heading2']))
            story.append(Paragraph(f"<b>User Request:</b> {query['user']}", styles['Normal']))
            story.append(Spacer(1, 0.1*inch))
            story.append(Paragraph(f"<b>Generated SQL:</b>", styles['Normal']))
            story.append(Paragraph(f"<font color='#4ECDC4'><code>{query['sql']}</code></font>", styles['Normal']))
            story.append(Spacer(1, 0.2*inch))

        doc.build(story)
        return f"✅ PDF Downloaded: {filename}", filename
    except Exception as e:
        return f"❌ Error: {str(e)}", None

In [13]:
def download_txt():
    if not query_history:
        return "❌ No queries to export!", None

    content = f"SQL Query History Report\nGenerated: {datetime.now()}\n{'='*60}\n\n"

    for i, query in enumerate(query_history, 1):
        content += f"Query #{i}\n"
        content += f"User Request: {query['user']}\n"
        content += f"Generated SQL: {query['sql']}\n"
        content += f"Timestamp: {query['timestamp']}\n"
        content += f"{'-'*60}\n\n"

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"sql_queries_{timestamp}.txt"

    try:
        Path(filename).write_text(content)
        return f"✅ TXT Downloaded: {filename}", filename
    except Exception as e:
        return f"❌ Error: {str(e)}", None

In [14]:
def generate_whatsapp_link():
    if not query_history:
        return "❌ No queries to share!"

    message = "🔍 SQL Queries Generated\n\n"
    for i, query in enumerate(query_history[-5:], 1):
        message += f"*Query {i}:* {query['user']}\n"
        message += f"```sql\n{query['sql']}\n```\n\n"

    encoded_message = urllib.parse.quote(message)
    whatsapp_link = f"https://wa.me/?text={encoded_message}"

    return f"✅ WhatsApp Link Created!\n\n[Click here to share on WhatsApp!]({whatsapp_link})"

In [15]:
def generate_facebook_link():
    if not query_history:
        return "❌ No queries to share!"

    message = "Check out these SQL queries I generated: "
    for query in query_history[-3:]:
        message += f"{query['user']} → {query['sql']} | "

    encoded_message = urllib.parse.quote(message)
    facebook_link = f"https://www.facebook.com/sharer/sharer.php?quote={encoded_message}"

    return f"✅ Facebook Link Created!\n\n[Click here to share on Facebook!]({facebook_link})"

print("🚀 Loading SQL Query Generator...")

🚀 Loading SQL Query Generator...


In [26]:
def clear_chat():
    global query_history
    query_history = []
    return []

with gr.Blocks(
    title="SQL Query Generator"
) as demo:

    gr.Markdown("""
    # 🔍 SQL Query Generator
    ## Natural Language → SQLite SQL

    Convert plain English requests into safe, production-ready SQL queries!
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="💬 Query Conversion",
                height=450,
                show_copy_button=True, # Reverting to this to avoid TypeError
                type='messages',
                allow_tags=False
            )

        with gr.Row():
            query_input = gr.Textbox(
                label="📝 Natural Language Request",
                placeholder="e.g., Show me all users who signed up last month",
                lines=2,
                scale=4
            )
            send_btn = gr.Button("Generate SQL", scale=1, size="lg", variant="primary")

            # Modified process_query to handle dictionary format for chat_history
            def process_query(user_query, chat_history):
                """Process database query request"""

                if not user_query.strip():
                    return chat_history

                # Gradio already adds the user message, so we just add the assistant's placeholder
                chat_history.append({'role': 'assistant', 'content': 'Processing...'})

                sql_query, explanation = generate_sql(user_query)

                if sql_query is None:
                    response_content = f"❌ ERROR\n\n{explanation}"
                else:
                    response_content = f"✅ SQL GENERATED\n\n```sql\n{sql_query}\n```\n\n📝 {explanation}"
                    query_history.append({
                        'user': user_query,
                        'sql': sql_query,
                        'timestamp': datetime.now().isoformat()
                    })

                # Update the last assistant message with the final response
                chat_history[-1]['content'] = response_content
                return chat_history

            send_btn.click(process_query, inputs=[query_input, chatbot], outputs=[chatbot])
            query_input.submit(process_query, inputs=[query_input, chatbot], outputs=[chatbot])

        with gr.Column(scale=1):
            gr.Markdown("""
            ### 🎮 Controls
            """)

    clear_btn = gr.Button("🗑️ Clear History", variant="stop", size="sm")
    clear_btn.click(clear_chat, outputs=[chatbot])

    gr.Markdown("---")
    gr.Markdown("### 📥 Export")

    pdf_btn = gr.Button("📄 Download PDF", size="sm", variant="primary")
    txt_btn = gr.Button("📋 Download TXT", size="sm", variant="primary")

    pdf_file = gr.File(label="PDF File", visible=True)
    txt_file = gr.File(label="TXT File", visible=True)

    export_status = gr.Textbox(label="Status", interactive=False, lines=2)

    pdf_btn.click(download_pdf, outputs=[export_status, pdf_file])
    txt_btn.click(download_txt, outputs=[export_status, txt_file])

    gr.Markdown("---")
    gr.Markdown("### 📱 Share")

    share_output = gr.Markdown(label="Share Link") # Changed to gr.Markdown

    whatsapp_btn = gr.Button("💬 WhatsApp", size="sm", variant="secondary")
    facebook_btn = gr.Button("👍 Facebook", size="sm", variant="secondary")

    whatsapp_btn.click(generate_whatsapp_link, outputs=[share_output])
    facebook_btn.click(generate_facebook_link, outputs=[share_output])

    gr.Markdown("---")
    gr.Markdown("""
    ### 📊 Schema Info

    *Tables:*
    - users
    - orders
    - products
    - transactions

    *Supported Requests:*
    - User queries (last month, etc)
    - Order lookups
    - Product searches
    - Transaction reports
    """)

    print("\n✅ SQL Query Generator Ready!\n")
    demo.launch(share=True, show_error=True)

/tmp/ipykernel_361/3449365655.py:19: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_361/3449365655.py:19: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(



✅ SQL Query Generator Ready!

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f14bdc5c8e8a2e736c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
